# R3-N2 LR latency benchmark

Benchmarks the approved R3-N1 calibrated LR-B bundle on the current Colab CPU. This notebook does not fit or alter the model, calibration, threshold, or bundle. Fresh subprocess startup is reported separately; it is not a cold VM measurement.

Select already-approved recordings in the fixture cell. The notebook does not inventory or search the dataset.

In [3]:
# Fresh-Runtime Cold-Start Benchmark — execute this cell first after restarting Colab.
# Exactly one first prediction is attempted. No warm-up or regular benchmark call occurs here.

import time
COLD_START_NS = time.perf_counter_ns()

COLD_ERRORS = []
COLD_PHASE = 'initialization'
ROOT = None
COLD_BUNDLE = None

COLD_MODEL_INIT_NS = None
COLD_BUNDLE_LOAD_NS = None
COLD_PREDICTION_NS = None
COLD_TOTAL_NS = None

COLD_PREDICTION_CALLS = 0
COLD_RESULT = None
COLD_DEVICE = None
COLD_IDENTITY = None
COLD_REVIEW = None
COLD_BUNDLE_LOAD_SUCCEEDED = False
COLD_FIXTURE = None
COLD_DURATION = None

try:
    import json, os, sys, platform, hashlib, math
    import importlib.metadata as metadata
    from pathlib import Path
    from datetime import datetime, timezone
    from google.colab import drive

    COLD_CREATED_UTC = datetime.now(timezone.utc).isoformat()

    COLD_PHASE = 'Google Drive mount'
    mount_ns = time.perf_counter_ns()

    if not Path('/content/drive/MyDrive').is_dir():
        drive.mount('/content/drive')

    DRIVE_MOUNT_NS = time.perf_counter_ns() - mount_ns

    COLD_PHASE = 'project path resolution'
    path_ns = time.perf_counter_ns()

    ROOT_CANDIDATES = [
        Path('/content/drive/MyDrive/PAI&AIES Project'),
        Path('/content/drive/MyDrive/EchoProof')
    ]

    ROOT = next(
        (p for p in ROOT_CANDIDATES if (p / 'src/echoproof/inference.py').is_file()),
        None
    )

    if ROOT is None:
        raise FileNotFoundError(
            'EchoProof root not found; edit ROOT_CANDIDATES to the Drive project folder'
        )

    os.chdir(ROOT)

    COLD_BUNDLE = ROOT / 'runs/r3/r3_n1_lr/selected_lr_v1/handoff/bundle'

    if not (COLD_BUNDLE / 'bundle.json').is_file():
        raise FileNotFoundError(f'Bundle missing: {COLD_BUNDLE}')

    sys.path.insert(0, str(ROOT / 'src'))
    PATH_RESOLUTION_NS = time.perf_counter_ns() - path_ns

    # Same fixture paths as the existing regular benchmark fixture cell.
    COLD_START_FIXTURES = {
        1: ROOT / 'dataset/genuine/wav/19-198-0000.wav',
        4: ROOT / 'dataset/genuine/wav/19-198-0037.wav',
        6: ROOT / 'dataset/genuine/wav/89-218-0004.wav',
        10: ROOT / 'dataset/genuine/wav/19-198-0002.wav',
        15: ROOT / 'dataset/genuine/wav/911-128684-0002.wav',
    }

    COLD_START_DURATION_SECONDS = 1

    if COLD_START_DURATION_SECONDS not in COLD_START_FIXTURES:
        raise ValueError('Choose an existing COLD_START_FIXTURES key')

    COLD_DURATION = COLD_START_DURATION_SECONDS
    COLD_FIXTURE = Path(COLD_START_FIXTURES[COLD_DURATION])

    if not COLD_FIXTURE.is_file():
        raise FileNotFoundError(f'Fixture missing: {COLD_FIXTURE}')

    # Import and initialize model libraries.
    COLD_PHASE = 'model/library initialization'
    model_init_ns = time.perf_counter_ns()

    import numpy, scipy, soundfile as sf, sklearn, joblib, librosa
    from echoproof.artifacts import file_digest
    from echoproof.inference import analyze, load_lr_bundle
    from echoproof.models.lr_detector import LRDetector, chunk_features
    from echoproof.models.r3_n1_lr_bundle import load_calibrated_lr_bundle

    COLD_MODEL_INIT_NS = time.perf_counter_ns() - model_init_ns

    # Prepare exactly one audio fixture.
    COLD_PHASE = 'fixture preparation'
    fixture_ns = time.perf_counter_ns()

    import io

    with sf.SoundFile(io.BytesIO(COLD_FIXTURE.read_bytes())) as audio:
        if audio.samplerate * COLD_DURATION > audio.frames:
            raise ValueError(f'Fixture shorter than {COLD_DURATION} seconds')

        samples = audio.read(
            frames=int(audio.samplerate * COLD_DURATION),
            dtype='float32',
            always_2d=True
        )
        rate = audio.samplerate

    audio_buffer = io.BytesIO()
    sf.write(
        audio_buffer,
        samples,
        rate,
        format='WAV',
        subtype='PCM_16'
    )
    audio_bytes = audio_buffer.getvalue()

    FIXTURE_PREPARATION_NS = time.perf_counter_ns() - fixture_ns

    # Load the production bundle.
    COLD_PHASE = 'bundle loading'
    load_ns = time.perf_counter_ns()

    loaded = load_lr_bundle(COLD_BUNDLE)

    COLD_BUNDLE_LOAD_NS = time.perf_counter_ns() - load_ns

    if loaded.spec.model_version != 'logistic_regression:B':
        raise ValueError(f'Unexpected model version: {loaded.spec.model_version}')

    COLD_REVIEW = 'approved R3-N1 LR-B bundle'
    COLD_BUNDLE_LOAD_SUCCEEDED = True

    # Exactly one production prediction. No warm-up.
    COLD_PHASE = 'first audio prediction'
    COLD_PREDICTION_CALLS = 1
    prediction_ns = time.perf_counter_ns()

    result = analyze(audio_bytes, loaded).result

    COLD_PREDICTION_NS = time.perf_counter_ns() - prediction_ns
    COLD_TOTAL_NS = time.perf_counter_ns() - COLD_START_NS

    COLD_RESULT = {
        'status': result.status,
        'score': result.score,
        'assessment': result.assessment,
        'model_version': result.model_version,
        'valid': bool(
            result.status == 'success'
            and result.score is not None
            and math.isfinite(float(result.score))
            and 0 <= float(result.score) <= 1
            and result.assessment in ('genuine', 'synthetic')
            and result.model_version == 'logistic_regression:B'
        )
    }

    if not COLD_RESULT['valid']:
        COLD_ERRORS.append({
            'phase': COLD_PHASE,
            'type': 'InvalidPrediction',
            'message': 'Single production prediction failed validation'
        })

except Exception as exc:
    COLD_TOTAL_NS = time.perf_counter_ns() - COLD_START_NS

    COLD_ERRORS.append({
        'phase': COLD_PHASE,
        'type': type(exc).__name__,
        'message': str(exc)
    })

    print(
        f'Cold-start measurement failed during {COLD_PHASE}: '
        f'{type(exc).__name__}: {exc}',
        flush=True
    )


# Capture environment and bundle hashes after prediction.
# Hashing cannot pre-read model weights before the prediction.
if ROOT is not None:
    try:
        packages = {}

        for name in [
            'numpy', 'scipy', 'soundfile', 'scikit-learn',
            'joblib', 'librosa', 'torch', 'echoproof'
        ]:
            try:
                packages[name] = metadata.version(name)
            except metadata.PackageNotFoundError:
                packages[name] = None

        COLD_DEVICE = {
            'platform': platform.platform(),
            'machine': platform.machine(),
            'processor': platform.processor(),
            'cpu_count_logical': os.cpu_count(),
            'python': sys.version,
            'packages': packages,
            'device': 'CPU (LRDetector NumPy inference)',
            'cuda_visible_devices': os.environ.get('CUDA_VISIBLE_DEVICES'),
        }

        try:
            with open('/proc/cpuinfo') as f:
                COLD_DEVICE['cpuinfo_model_name'] = next(
                    (
                        line.split(':', 1)[1].strip()
                        for line in f
                        if line.lower().startswith('model name')
                    ),
                    None
                )
        except OSError:
            COLD_DEVICE['cpuinfo_model_name'] = None

        try:
            COLD_DEVICE['colab_release_file'] = Path(
                '/etc/colab-release'
            ).read_text(errors='replace').strip()
        except OSError:
            COLD_DEVICE['colab_release_file'] = None

        hash_ns = time.perf_counter_ns()

        files = sorted(
            p for p in COLD_BUNDLE.rglob('*') if p.is_file()
        )

        COLD_IDENTITY = {
            'bundle_path': str(COLD_BUNDLE),
            'bundle_json_sha256': file_digest(COLD_BUNDLE / 'bundle.json'),
            'files': {
                p.relative_to(COLD_BUNDLE).as_posix(): file_digest(p)
                for p in files
            }
        }

        BUNDLE_IDENTITY_CAPTURE_NS = time.perf_counter_ns() - hash_ns

    except Exception as exc:
        COLD_ERRORS.append({
            'phase': 'post-measurement metadata',
            'type': type(exc).__name__,
            'message': str(exc)
        })
        BUNDLE_IDENTITY_CAPTURE_NS = None

    stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')

    report_path = (
        ROOT / 'reports/r3' /
        f'r3_n2_lr_fresh_runtime_cold_start_{stamp}.json'
    )

    report = {
        'schema': 'r3-n2-lr-fresh-runtime-cold-start-v1',
        'model': 'LR-B production bundle',
        'benchmark_type': 'fresh-runtime cold-start; single first prediction; not cold VM start',
        'created_utc': locals().get('COLD_CREATED_UTC'),
        'project_root': str(ROOT),

        'timing_scope': {
            'clock': 'time.perf_counter_ns',
            'prediction_calls': COLD_PREDICTION_CALLS,
            'warmup_calls': 0,
            'vm_provisioning_time_included': False,
            'note': (
                'Colab VM provisioning is excluded because notebook execution '
                'begins only after the runtime is available.'
            )
        },

        'device': COLD_DEVICE,
        'bundle': COLD_IDENTITY,
        'bundle_review_status': COLD_REVIEW,
        'bundle_load_succeeded': COLD_BUNDLE_LOAD_SUCCEEDED,

        'fixture': {
            'path': str(COLD_FIXTURE) if COLD_FIXTURE else None,
            'duration_seconds': COLD_DURATION
        },

        'measurements_ns': {
            'cell_start_to_first_prediction_total': COLD_TOTAL_NS,
            'model_library_initialization': COLD_MODEL_INIT_NS,
            'bundle_loading': COLD_BUNDLE_LOAD_NS,
            'first_audio_prediction': COLD_PREDICTION_NS,
            'drive_mount': locals().get('DRIVE_MOUNT_NS'),
            'project_path_resolution': locals().get('PATH_RESOLUTION_NS'),
            'fixture_preparation': locals().get('FIXTURE_PREPARATION_NS'),
            'post_measurement_bundle_identity_capture': locals().get(
                'BUNDLE_IDENTITY_CAPTURE_NS'
            )
        },

        'prediction': COLD_RESULT,
        'errors': COLD_ERRORS
    }

    try:
        report_path.parent.mkdir(parents=True, exist_ok=True)

        report_path.write_text(
            json.dumps(report, indent=2, ensure_ascii=False, default=str) + '\n',
            encoding='utf-8'
        )

        print('Fresh-runtime cold-start report:', report_path)
        print(
            'Prediction valid:',
            None if COLD_RESULT is None else COLD_RESULT['valid']
        )
        print(json.dumps(report['measurements_ns'], indent=2))

    except Exception as exc:
        print(
            f'Could not save cold-start report: {type(exc).__name__}: {exc}',
            flush=True
        )

else:
    print('Report not saved: project root could not be resolved.', flush=True)

Mounted at /content/drive
Fresh-runtime cold-start report: /content/drive/MyDrive/PAI&AIES Project/reports/r3/r3_n2_lr_fresh_runtime_cold_start_20261114T115804Z.json
Prediction valid: True
{
  "cell_start_to_first_prediction_total": 144711793581,
  "model_library_initialization": 8103012595,
  "bundle_loading": 1132564997,
  "first_audio_prediction": 30195662547,
  "drive_mount": 16453670365,
  "project_path_resolution": 2101015570,
  "fixture_preparation": 542051203,
  "post_measurement_bundle_identity_capture": 20938590
}


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import os
CANDIDATE_ROOTS = [Path('/content/drive/MyDrive/PAI&AIES Project')]
ROOT = next((p for p in CANDIDATE_ROOTS if (p / 'src/echoproof/inference.py').is_file()), None)
if ROOT is None: raise FileNotFoundError('EchoProof root not found; edit CANDIDATE_ROOTS to your Drive project folder')
os.chdir(ROOT)
BUNDLE = ROOT / 'runs/r3/r3_n1_lr/selected_lr_v1/handoff/bundle'
if not (BUNDLE / 'bundle.json').is_file(): raise FileNotFoundError(f'Approved bundle missing: {BUNDLE}')
print('Project root:', ROOT)
print('Bundle:', BUNDLE)

## Editable fixture selection

Add explicit paths to existing recordings already approved for this benchmark. Each path must supply the target duration; the benchmark uses the first N seconds in memory and never writes a crop. Missing groups are recorded as missing, never silently omitted. No directory scan occurs.

In [ ]:
# EDIT THESE five entries to point at existing approved recordings.
FIXTURES = {
    1:  ROOT / 'dataset/genuine/wav/19-198-0000.wav',
    4:  ROOT / 'dataset/genuine/wav/19-198-0037.wav',
    6: ROOT / 'dataset/genuine/wav/89-218-0004.wav',
    10: ROOT / 'dataset/genuine/wav/19-198-0002.wav',
    15: ROOT / 'dataset/genuine/wav/911-128684-0002.wav',
}
WARMUP_CALLS = 5
MEASURED_CALLS = 100
TOLERANCE = 1e-10
# Paths can be WAV/FLAC supported by production decode. Provide a custom explicit path per duration.

In [ ]:
import sys
sys.path.insert(0, str(ROOT / 'src'))

import sys, platform, subprocess, json, time, hashlib, importlib.metadata as metadata
import numpy as np
import soundfile as sf
from datetime import datetime, timezone
from echoproof.inference import load_lr_bundle, analyze
from echoproof.artifacts import file_digest
from echoproof.models.lr_detector import LRDetector, chunk_features
import echoproof.inference as inference
import echoproof.models.lr_detector as lr_detector
import echoproof.data.chunks as chunk_module

def package_versions():
    names = ['numpy','scipy','soundfile','scikit-learn','joblib','librosa','torch','echoproof']
    out = {}
    for name in names:
        try: out[name] = metadata.version(name)
        except metadata.PackageNotFoundError: out[name] = None
    return out

def device_details():
    details = {'platform': platform.platform(), 'machine': platform.machine(), 'processor': platform.processor(), 'cpu_count_logical': os.cpu_count(), 'python': sys.version, 'packages': package_versions(), 'device': 'CPU (LRDetector NumPy inference)', 'cuda_visible_devices': os.environ.get('CUDA_VISIBLE_DEVICES')}
    try:
        with open('/proc/cpuinfo') as f: details['cpuinfo_model_name'] = next((line.split(':',1)[1].strip() for line in f if line.lower().startswith('model name')), None)
    except OSError: details['cpuinfo_model_name'] = None
    try: details['colab_release_file'] = Path('/etc/colab-release').read_text(errors='replace').strip()
    except OSError: details['colab_release_file'] = None
    return details

def percentile(values, q): return float(np.percentile(np.asarray(values, dtype=np.float64), q)) if values else None
def digest_bundle():
    paths = sorted(p for p in BUNDLE.rglob('*') if p.is_file())
    return {'bundle_path': str(BUNDLE), 'bundle_json_sha256': file_digest(BUNDLE/'bundle.json'), 'files': {p.relative_to(BUNDLE).as_posix(): file_digest(p) for p in paths}}

bundle_identity = digest_bundle()
loaded = load_lr_bundle(BUNDLE)
assert loaded.spec.model_version == 'logistic_regression:B'
print('Bundle identity captured; approved production loader succeeded.')
print(json.dumps(device_details(), indent=2))

## Benchmark and parity

Normal production `analyze()` is the reference. Stage instrumentation wraps the production decode/window/scorer/aggregation/calibration/decision callables temporarily and restores them after each measured call. Feature extraction and LR prediction are timed separately inside the same production LR methods. Total and raw per-call nanosecond samples are preserved.

In [2]:
def instrumented_analyze(audio_bytes, bundle):
    # LRDetector.score_chunks calls lr_detector.chunk_features(waveforms), then
    # self.predict_proba_features(features); these wrappers instrument those exact production calls.
    stages = {k: 0 for k in ['decode','chunk_construction','feature_extraction','lr_prediction','aggregation','calibration','threshold_decision','total']}
    calls = {k: 0 for k in stages}
    original_spec = bundle.spec
    original_decision_rule = original_spec.decision_rule
    originals = {
        'decode_audio': inference.decode_audio,
        'eval_windows': inference.eval_windows,
        'scorer': bundle.scorer,
        'aggregate': bundle.aggregate,
        'calibrate': bundle.calibrate,
        'spec': original_spec,
        'decision_rule': original_decision_rule,
        'chunk_features': lr_detector.chunk_features,
        'predict_proba_features': LRDetector.predict_proba_features,
    }
    def timed(key, fn):
        def call(*args, **kwargs):
            start_ns = time.perf_counter_ns()
            calls[key] += 1
            try:
                return fn(*args, **kwargs)
            finally:
                stages[key] += time.perf_counter_ns() - start_ns
        return call
    try:
        inference.decode_audio = timed('decode', originals['decode_audio'])
        inference.eval_windows = timed('chunk_construction', originals['eval_windows'])
        lr_detector.chunk_features = timed('feature_extraction', originals['chunk_features'])
        LRDetector.predict_proba_features = timed('lr_prediction', originals['predict_proba_features'])
        object.__setattr__(
        bundle, 'aggregate',
         timed('aggregation', originals['aggregate'])
        )
        if originals['calibrate'] is not None:
             object.__setattr__(
             bundle, 'calibrate',
             timed('calibration', originals['calibrate'])
             )
        class DecisionProxy:
            def decide(self, score):
                return timed('threshold_decision', originals['decision_rule'].decide)(score)
        object.__setattr__(original_spec, 'decision_rule', DecisionProxy())
        start_ns = time.perf_counter_ns()
        result = analyze(audio_bytes, bundle)
        stages['total'] = time.perf_counter_ns() - start_ns
        calls['total'] += 1
        if calls['feature_extraction'] == 0 or calls['lr_prediction'] == 0:
            raise RuntimeError('production scorer did not invoke feature extraction and LR prediction wrappers')
        return result, stages, calls
    finally:
        inference.decode_audio = originals['decode_audio']
        inference.eval_windows = originals['eval_windows']
        lr_detector.chunk_features = originals['chunk_features']
        LRDetector.predict_proba_features = originals['predict_proba_features']
        object.__setattr__(bundle, 'aggregate', originals['aggregate'])
        object.__setattr__(bundle, 'calibrate', originals['calibrate'])
        object.__setattr__(original_spec, 'decision_rule', originals['decision_rule'])


def prefix_bytes(path, seconds):
    # In-memory prefix for one explicit fixture; no crop file is written.
    import soundfile as sf, io
    data = Path(path).read_bytes()
    with sf.SoundFile(io.BytesIO(data)) as f:
        if f.samplerate * seconds > f.frames:
            raise ValueError(f'fixture is shorter than {seconds}s')
        samples = f.read(frames=int(f.samplerate*seconds), dtype='float32', always_2d=True)
        rate = f.samplerate
    buf = io.BytesIO()
    sf.write(buf, samples, rate, format='WAV', subtype='PCM_16')
    return buf.getvalue()


def fresh_process(path, duration):
    # Each call launches an independent interpreter. The child measures bundle loading
    # and its first production prediction separately; the parent measures spawn-to-result.
    child = 'import json,sys,time\nfrom pathlib import Path\nsys.path.insert(0, str(Path(sys.argv[1]) / "src"))\nfrom echoproof.inference import load_lr_bundle,analyze\nimport soundfile as sf,io\nroot,bundle,audio_path,dur=sys.argv[1],sys.argv[2],sys.argv[3],int(sys.argv[4])\ndata=Path(audio_path).read_bytes()\nwith sf.SoundFile(io.BytesIO(data)) as f: x=f.read(frames=int(f.samplerate*dur),dtype="float32",always_2d=True); rate=f.samplerate\nb=io.BytesIO(); sf.write(b,x,rate,format="WAV",subtype="PCM_16"); raw=b.getvalue()\nt=time.perf_counter_ns(); model=load_lr_bundle(bundle); load_ns=time.perf_counter_ns()-t\nt=time.perf_counter_ns(); result=analyze(raw,model); first_ns=time.perf_counter_ns()-t\nprint(json.dumps({"bundle_load_ns":load_ns,"first_prediction_ns":first_ns,"status":result.result.status,"score":result.result.score,"assessment":result.result.assessment,"model_version":result.result.model_version}))'
    start_ns = time.perf_counter_ns()
    proc = subprocess.run([sys.executable, '-c', child, str(ROOT), str(BUNDLE), str(path), str(duration)], capture_output=True, text=True, timeout=180, env={**os.environ, 'PYTHONPATH':str(ROOT/ 'src')})
    process_to_result_ns = time.perf_counter_ns() - start_ns
    if proc.returncode:
        raise RuntimeError(proc.stderr[-2000:])
    return {'process_to_first_prediction_ns': process_to_result_ns, **json.loads(proc.stdout.strip().splitlines()[-1])}


FRESH_PROCESS_REPETITIONS = 10
report = {'schema':'r3-n2-lr-latency-v1','created_utc':datetime.now(timezone.utc).isoformat(),'project_root':str(ROOT),'benchmark_kind':'fresh subprocess startup; not cold VM start','device':device_details(),'bundle':bundle_identity,'configuration':{'batch_size':1,'warmup_calls':WARMUP_CALLS,'measured_calls':MEASURED_CALLS,'fresh_process_repetitions':FRESH_PROCESS_REPETITIONS,'durations_seconds': sorted(FIXTURES.keys()),'clock':'time.perf_counter_ns','tolerance':TOLERANCE},'groups':{},'errors':[]}
for seconds, path in FIXTURES.items():
    key = str(seconds)
    group = {'duration_seconds':seconds,'fixture_path':str(path),'warmup_raw_ns':[],'raw_samples_ns':[],'stage_samples_ns':{},'fresh_process_repetitions':[],'fresh_process_errors':[],'errors':[],'timeouts':0}
    report['groups'][key] = group
    if not Path(path).is_file():
        group['status']='missing_fixture'
        group['errors'].append('fixture path missing or not edited')
        print(f'MISSING {seconds}s fixture: {path}', flush=True)
        continue
    try:
        raw = prefix_bytes(path, seconds)
        group['actual_input_duration_seconds'] = seconds
        ref = analyze(raw, loaded)
        group['production_reference'] = {'status':ref.result.status,'score':ref.result.score,'assessment':ref.result.assessment,'model_version':ref.result.model_version}
        for repetition in range(FRESH_PROCESS_REPETITIONS):
            try:
                fresh = fresh_process(path, seconds)
                group['fresh_process_repetitions'].append(fresh)
            except subprocess.TimeoutExpired as exc:
                group['timeouts'] += 1
                group['fresh_process_errors'].append({'repetition':repetition+1,'type':'TimeoutExpired','message':str(exc)})
            except Exception as exc:
                group['fresh_process_errors'].append({'repetition':repetition+1,'type':type(exc).__name__,'message':str(exc)})
            print(f'{seconds}s fresh process {repetition+1}/{FRESH_PROCESS_REPETITIONS}', flush=True)
        fresh_metrics = {}
        for metric in ['process_to_first_prediction_ns','bundle_load_ns','first_prediction_ns']:
            vals = [int(item[metric]) for item in group['fresh_process_repetitions'] if item.get(metric) is not None]
            fresh_metrics[metric] = {'count':len(vals),'p50':percentile(vals,50),'p95':percentile(vals,95),'raw_samples_ns':vals}
        group['fresh_process_summary_ns'] = fresh_metrics
        for i in range(WARMUP_CALLS):
            _, stages, _ = instrumented_analyze(raw, loaded)
            group['warmup_raw_ns'].append(int(stages['total']))
            print(f'{seconds}s warmup {i+1}/{WARMUP_CALLS}', flush=True)
        parity_scores=[]
        parity_decisions=[]
        for i in range(MEASURED_CALLS):
            got, stages, calls = instrumented_analyze(raw, loaded)
            score_delta = abs(float(got.result.score)-float(ref.result.score)) if got.result.score is not None and ref.result.score is not None else None
            parity_scores.append(score_delta)
            parity_decisions.append(got.result.assessment == ref.result.assessment)
            group['raw_samples_ns'].append(int(stages['total']))
            for name, value in stages.items():
                group['stage_samples_ns'].setdefault(name,[]).append(int(value))
            if calls['feature_extraction'] < 1 or calls['lr_prediction'] < 1:
                raise RuntimeError('feature extraction or LR prediction instrumentation wrapper was not invoked')
            print(f'{seconds}s measured {i+1}/{MEASURED_CALLS}', flush=True)
        group['parity']={'reference_match':bool(all(x is not None and x <= TOLERANCE for x in parity_scores)),'max_abs_score_delta':max((x for x in parity_scores if x is not None),default=None),'decision_flips':sum(not x for x in parity_decisions),'checks':len(parity_scores)}
        group['status']='completed' if len(group['fresh_process_repetitions']) == FRESH_PROCESS_REPETITIONS else 'completed_with_fresh_process_errors'
        group['summary_ns']={name:{'count':len(vals),'p50':percentile(vals,50),'p95':percentile(vals,95),'mean':float(np.mean(vals)) if vals else None} for name,vals in group['stage_samples_ns'].items()}
    except Exception as exc:
        group['status']='error'
        group['errors'].append(f'{type(exc).__name__}: {exc}')
        print(f'ERROR {seconds}s: {exc}',flush=True)

OUTPUT = ROOT/'reports/r3'
OUTPUT.mkdir(parents=True,exist_ok=True)
stamp=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
REPORT_PATH=OUTPUT/f'r3_n2_lr_latency_{stamp}.json'
REPORT_PATH.write_text(json.dumps(report,indent=2,ensure_ascii=False,default=str)+'\n',encoding='utf-8')
print('Saved report:',REPORT_PATH)
print('Status by duration:',{k:v['status'] for k,v in report['groups'].items()})


NameError: name 'datetime' is not defined

In [ ]:

import json
from pathlib import Path
from datetime import datetime, timezone

# Use the report generated by the benchmark
source = Path(REPORT_PATH)

with source.open("r", encoding="utf-8") as f:
    report = json.load(f)

summary = {
    "schema": "r3-n2-lr-latency-summary-v1",
    "source_report": str(source),
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "benchmark_kind": report.get("benchmark_kind"),
    "device": report.get("device"),
    "bundle": report.get("bundle"),
    "configuration": report.get("configuration"),
    "groups": {},
    "overall": {
        "duration_groups": len(report.get("groups", {})),
        "fresh_process_attempts": 0,
        "fresh_process_successes": 0,
        "fresh_process_failures": 0,
        "warm_measurements": 0,
        "parity_checks": 0,
        "parity_failures": 0
    },
    "top_level_errors": report.get("errors", [])
}

for duration, group in report.get("groups", {}).items():
    fresh = group.get("fresh_process_repetitions", [])
    fresh_errors = group.get("fresh_process_errors", [])
    parity = group.get("parity", {})
    warm = group.get("summary_ns", {})

    summary["groups"][duration] = {
        "status": group.get("status"),
        "fixture_path": group.get("fixture_path"),
        "actual_input_duration_seconds": group.get(
            "actual_input_duration_seconds"
        ),
        "warm_measurements": len(group.get("raw_samples_ns", [])),
        "warm_latency_ms": {
            stage: {
                "count": values.get("count"),
                "p50": (
                    values["p50"] / 1e6
                    if values.get("p50") is not None else None
                ),
                "p95": (
                    values["p95"] / 1e6
                    if values.get("p95") is not None else None
                ),
                "mean": (
                    values["mean"] / 1e6
                    if values.get("mean") is not None else None
                )
            }
            for stage, values in warm.items()
        },
        "parity": parity,
        "fresh_process": {
            "attempts": len(fresh) + len(fresh_errors),
            "successes": len(fresh),
            "failures": len(fresh_errors),
            "errors": fresh_errors,
            "latency_ms": {
                metric: {
                    "count": values.get("count"),
                    "p50": (
                        values["p50"] / 1e6
                        if values.get("p50") is not None else None
                    ),
                    "p95": (
                        values["p95"] / 1e6
                        if values.get("p95") is not None else None
                    )
                }
                for metric, values in group.get(
                    "fresh_process_summary_ns", {}
                ).items()
            }
        },
        "group_errors": group.get("errors", [])
    }

    summary["overall"]["fresh_process_attempts"] += (
        len(fresh) + len(fresh_errors)
    )
    summary["overall"]["fresh_process_successes"] += len(fresh)
    summary["overall"]["fresh_process_failures"] += len(fresh_errors)
    summary["overall"]["warm_measurements"] += len(
        group.get("raw_samples_ns", [])
    )
    summary["overall"]["parity_checks"] += parity.get("checks", 0)

    if not parity.get("reference_match", False):
        summary["overall"]["parity_failures"] += 1

# Save compact summary beside the original report
summary_path = source.with_name(
    source.stem + "_summary.json"
)

summary_path.write_text(
    json.dumps(summary, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("SUMMARY FILE:", summary_path)
print(json.dumps(summary, indent=2, ensure_ascii=False))
